# D · Remaining Rounds + Time to Next Action (LLM rows)

**Before you start:** the `D_trajectory` folder in My Drive must contain the new files (`code/run_rounds_next_llm.py`,
`code/build_rounds_next_data.py`, `code/baselines_rounds_next.py`, `code/score_rounds_next.py`,
`code/build_forecasting_table.py`, `prompts/rounds_next_forecast.txt`). `data_v1_share_v1.2.zip` must be in My Drive too.

1. Choose the models in **Step 0** (`RUN_GPT`, `RUN_QWEN`). For Qwen, start the Azure deployment first.
2. Secrets (key icon on the left, **Notebook access** on). GPT: `OPENAI_API_KEY` (OpenAI's own API; used first if present),
   or the three Azure ones `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_DEPLOYMENT`.
   Qwen: `QWEN_BASE_URL`, `QWEN_AZURE_API_KEY` (optional `QWEN_DEPLOYMENT`).
3. **Runtime → Run all.** Each model runs 3 dev cases (a check, not scored), then the 100 test_core cases twice:
   with A's A1 state + history, and history only. These are the same cases, inputs and settings as the Disposition runs.
   One call per case answers both tasks.
4. If Colab disconnects, just **Run all** again. Finished cases are saved and are not paid for twice.
5. When it says **ALL DONE**, delete the Qwen deployment so billing stops.

In [ ]:
# Step 0: which models to run
RUN_GPT = True    # GPT-5-mini, reasoning_effort low (about 40-60 min, about $3-5)
RUN_QWEN = False  # Qwen3.6-27B team setting: needs a live Azure deployment (billed per hour). Off while Azure is disabled.

In [ ]:
# Step 1: find the files on Drive, unzip the data, install the tools
from google.colab import drive
drive.mount('/content/drive')
import glob, os, subprocess, sys
roots = glob.glob('/content/drive/MyDrive/**/D_trajectory/code/run_trajectory_llm.py', recursive=True)
zips = glob.glob('/content/drive/MyDrive/**/data_v1_share_v1.2.zip', recursive=True)
assert roots, 'Cannot find the D_trajectory folder in My Drive.'
assert zips, 'Cannot find data_v1_share_v1.2.zip in My Drive.'
ROOT = os.path.dirname(os.path.dirname(roots[0]))
need = ['code/run_rounds_next_llm.py', 'code/build_rounds_next_data.py', 'code/baselines_rounds_next.py',
        'code/score_rounds_next.py', 'code/build_forecasting_table.py', 'prompts/rounds_next_forecast.txt',
        'inputs/state_predictions_A.jsonl', 'inputs/state_predictions_dev_C.jsonl', 'inputs/trajectory_inputs.jsonl']
missing = [n for n in need if not os.path.exists(os.path.join(ROOT, n))]
assert not missing, f'Missing in D_trajectory on Drive: {missing}'
DATA = '/content/data_v1_share'
if not os.path.exists(DATA + '/labels.csv'):
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/content/'], check=True)
print('ROOT =', ROOT); print('DATA =', DATA)
!pip -q install "openai==2.54.0" "scikit-survival==0.28.0"
%cd {ROOT}/code
print('Step 1 OK')

In [ ]:
# Step 2: rebuild the labels from data v1.2 and the KM / Cox rows (no API, free). They must match the frozen labels.
def sh(args):
    p = subprocess.Popen([sys.executable, '-u'] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end=''); out.append(line)
    p.wait()
    return p.returncode, ''.join(out)
rc, out = sh(['build_rounds_next_data.py', '--data', DATA, '--out', ROOT])
assert rc == 0 and 'remaining_oas 100/100' in out and 'target date 100/100' in out, 'Labels differ. Send the message above to Claude.'
rc, out = sh(['baselines_rounds_next.py', '--root', ROOT]); assert rc == 0
print('Step 2 OK')

In [ ]:
# Step 3: read the secrets (never typed into the notebook) and test the connection
from google.colab import userdata
def get(k):
    try:
        return (userdata.get(k) or '').strip()
    except Exception:
        return ''
def run(provider, args):
    return sh(['run_rounds_next_llm.py', '--provider', provider, '--root', ROOT, '--data', DATA] + args)
if RUN_GPT:
    if get('OPENAI_API_KEY'):   # OpenAI's own API, same GPT-5-mini model (used while the Azure subscription is disabled)
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY']:
            os.environ.pop(k, None)
        os.environ['OPENAI_API_KEY'] = get('OPENAI_API_KEY')
        os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-5-mini'
        print('GPT route: OpenAI API, model gpt-5-mini')
    else:
        for k in ['AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_KEY', 'AZURE_OPENAI_DEPLOYMENT']:
            assert get(k), f'Secret {k} is missing (or add OPENAI_API_KEY instead)'
            os.environ[k] = get(k)
        print('GPT route: Azure, deployment', os.environ['AZURE_OPENAI_DEPLOYMENT'])
    rc, out = run('azure', ['--ping']); assert rc in (0, None) and 'connection ok' in out, 'GPT test failed. Send the message above to Claude.'
if RUN_QWEN:
    for k in ['QWEN_BASE_URL', 'QWEN_AZURE_API_KEY']:
        assert get(k), f'Secret {k} is missing'
        os.environ[k] = get(k)
    os.environ['QWEN_DEPLOYMENT'] = get('QWEN_DEPLOYMENT') or 'qwen--qwen36-27b'
    rc, out = run('qwen', ['--ping']); assert 'connection ok' in out, 'Qwen test failed. Send the message above to Claude.'
print('Step 3 OK')

In [ ]:
# Step 4: dev check (3 cases, not scored), then test_core: state + history, then history only
STATE_DEV = os.path.join(ROOT, 'inputs', 'state_predictions_dev_C.jsonl')
STATE = os.path.join(ROOT, 'inputs', 'state_predictions_A.jsonl')
for provider, ok in (('azure', RUN_GPT), ('qwen', RUN_QWEN)):
    if not ok:
        continue
    w = '2'   # 2 at a time: stays under API rate limits
    rc, out = run(provider, ['--split', 'dev', '--workers', '1', '--state', STATE_DEV, '--state-version', 'C_dev_gpt5mini'])
    assert rc == 0 and 'ok 0/' not in out, f'{provider} dev run failed. Send the message above to Claude.'
    rc, out = run(provider, ['--split', 'core', '--workers', w, '--state', STATE, '--state-version', 'A_v1'])
    assert rc == 0, f'{provider} state run stopped. Send the message above to Claude.'
    rc, out = run(provider, ['--split', 'core', '--workers', w])
    assert rc == 0, f'{provider} history-only run stopped. Send the message above to Claude.'
print('Step 4 OK')

In [ ]:
# Step 4b (optional, Qwen only): if some cases failed only because of a timeout / connection error ("failed: ... api: ..."),
# change False to True and run this cell once. Failures stay in the log.
RETRY_FAILED = False
if RETRY_FAILED:
    run('qwen', ['--split', 'core', '--workers', '1', '--state', STATE, '--state-version', 'A_v1', '--retry-failed'])
    run('qwen', ['--split', 'core', '--workers', '1', '--retry-failed'])

In [ ]:
# Step 5: score everything and fill the paper's forecasting table
sh(['score_rounds_next.py', '--root', ROOT])
sh(['build_forecasting_table.py', '--root', ROOT])
import pandas as pd
display(pd.read_csv(f'{ROOT}/results/forecasting_table.csv').round(3))
print('ALL DONE. Take a screenshot of this page and send it to Claude. Delete the Qwen deployment now if you used it.')